### Time series datasets (DIF and IF comparison)

#### 1. Synthetic Financial Datasets from Kaggle

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from algorithms import DIF  
from sklearn.ensemble import IsolationForest
# import matplotlib.pyplot as plt

# Load the data
df = pd.read_csv('new_data/ts/PSlog.csv')

n = len(df)
df = df.iloc[:int(0.1 * n)]

df.drop(columns=['nameOrig', 'nameDest', 'isFlaggedFraud'], inplace=True)

df = pd.get_dummies(df, columns=['type'])

labels = df['isFraud'].values
df.drop(columns=['isFraud'], inplace=True)

scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(df)
X_scaled = X_scaled.astype(np.float32)

X_seq, y_seq = np.array(X_scaled), np.array(labels)

train_size = int(0.8 * len(X_seq))
X_train, X_test = X_seq[:train_size], X_seq[train_size:]
y_train, y_test = y_seq[:train_size], y_seq[train_size:]

In [ ]:
# import gc, torch
# gc.collect()
# try:
#     torch.cuda.empty_cache()
# except Exception:
#     pass


In [ ]:
# import os
# os.environ["CUDA_VISIBLE_DEVICES"] = ""     
# os.environ["OMP_NUM_THREADS"] = "1"       
# os.environ["MKL_NUM_THREADS"] = "1"

# import numpy as np
# np.set_printoptions(suppress=True)


In [2]:

def cap_sample(X, cap=50000, seed=0):
    
    if len(X) <= cap: 
        return X
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(X), size=cap, replace=False)
    return X[idx]

X_train = X_train.astype(np.float32, copy=False)
X_test  = X_test.astype(np.float32,  copy=False)

# Sample first and then perform PCA dimensionality reduction
X_small = cap_sample(X_train, cap=40000, seed=42)

sc = StandardScaler()
X_small = sc.fit_transform(X_small).astype(np.float32, copy=False)
X_train = sc.transform(X_train).astype(np.float32, copy=False)
X_test  = sc.transform(X_test).astype(np.float32,  copy=False)

pca = PCA(10, svd_solver="randomized", random_state=42)
pca.fit(X_small)  # Fit in small samples
X_train = pca.transform(X_train).astype(np.float32, copy=False)
X_test  = pca.transform(X_test).astype(np.float32,  copy=False)


In [ ]:
dif = DIF(
    n_ensemble=20,      # Decrease because of OOM
    n_estimators=6,  
    random_state=42,
)

dif.fit(X_train)

network additional parameters: {'n_hidden': [500, 100], 'n_emb': 20, 'skip_connection': None, 'dropout': None, 'activation': 'tanh', 'be_size': 20}


In [4]:
# Score in blocks to avoid OOM
def chunked_df(model, X, chunk=4096):
    out = np.empty((len(X),), dtype=np.float32)
    s = 0
    while s < len(X):
        e = min(s+chunk, len(X))
        out[s:e] = model.decision_function(X[s:e])
        s = e
    return out

score = chunked_df(dif, X_test, chunk=2048)

auc_dif = roc_auc_score(y_test, score)
pr_dif  = average_precision_score(y_test, score)
print(f"[DIF]     AUC-ROC={auc_dif:.4f}  AUC-PR={pr_dif:.4f}")

[DIF]     AUC-ROC=0.8335  AUC-PR=0.0074


In [ ]:
# IF
ifor = IsolationForest(
    n_estimators=120,  # Consistent with the total number of trees in DIF
    max_samples=256,
    contamination="auto",
    random_state=42,
    n_jobs=-1
)
ifor.fit(X_train)

score_if = -ifor.score_samples(X_test)

auc_if = roc_auc_score(y_test, score_if)
pr_if  = average_precision_score(y_test, score_if)
print(f"[IForest] AUC-ROC={auc_if:.4f}  AUC-PR={pr_if:.4f}")

[IForest] AUC-ROC=0.8740  AUC-PR=0.0486


In [10]:
print("\n== SUMMARY (Synthetic Financial Datasets ) ==")
print(f"[DIF]     AUC-ROC={auc_dif:.4f}  AUC-PR={pr_dif:.4f}")
print(f"[IForest] AUC-ROC={auc_if:.4f}  AUC-PR={pr_if:.4f}")


== SUMMARY (Synthetic Financial Datasets ) ==
[DIF]     AUC-ROC=0.8718  AUC-PR=0.0421
[IForest] AUC-ROC=0.8740  AUC-PR=0.0486


#### 2. Created datasets

In [ ]:

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.preprocessing import StandardScaler
from algorithms import DIF
from sklearn.ensemble import IsolationForest


# Create datasets
rng = np.random.default_rng(0)
T_train, T_test = 20000, 40000
D =3      # Dimension                          
fs = 1.0  # Sampling frequency
t_tr = np.arange(T_train) / fs
t_te = np.arange(T_test)  / fs

# Nonlinear background noise of multiple frequencies/phases + acoustic term + nonlinear activation
base_freqs = rng.uniform(0.005, 0.05, size=D)
phases     = rng.uniform(0, 2*np.pi, size=D)
amp        = rng.uniform(0.5, 1.3, size=D)

def synth(t):
    S = []
    for d in range(D):
        f = base_freqs[d]
        x = (amp[d] * np.sin(2*np.pi*f*t + phases[d])
             + 0.5*np.sin(2*np.pi*2.7*f*t + 0.3)
             + 0.3*np.sin(2*np.pi*0.7*f*t + 1.7))
        # Nonlinear distortion (slight)
        x = np.tanh(x) + 0.05*np.sin(2*np.pi*0.013*t)
        S.append(x)
    return np.stack(S, axis=1)

X_train = synth(t_tr) + rng.normal(0, 0.05, size=(T_train, D))
X_test = synth(t_te) + rng.normal(0, 0.05, size=(T_test,  D))

# Inject outliers
labels = np.zeros(T_test, dtype=np.int32)

def inject_segment(lo, hi, func):
    lo = max(0, lo); hi = min(T_test, hi)
    X_test[lo:hi] = func(X_test[lo:hi]); labels[lo:hi] = 1

# Abnormal shape: Short-time multi-channel square wave/sawtooth replacement
for s in [800]:
    width = 200
    k = min(D, 10)  
    idx = rng.choice(D, size=k, replace=False)
    def shape_anom(seg):
        tt = np.arange(seg.shape[0])
        sq = np.sign(np.sin(2*np.pi*0.12*tt))  # square wave
        saw = 2*(tt/len(tt) - np.floor(0.5 + tt/len(tt)))
        seg[:, idx] = 0.7*seg[:, idx] + 1.2*sq[:, None] + 0.8*saw[:, None]
        return seg
    inject_segment(s, s+width, shape_anom)

# Phase hits
for s in [1200]:
    width = 100
    k = min(D, 8) 
    idx = rng.choice(D, size=k, replace=False)
    def phase_flip(seg):
        tt = np.arange(seg.shape[0])
        flip = np.sin(2*np.pi*0.2*tt + np.pi)  # Phase π
        seg[:, idx] = 0.6*seg[:, idx] + 1.1*flip[:, None]
        return seg
    inject_segment(s, s+width, phase_flip)

# Frequency discontinuity
for s in [1500]:
    width = 100
    k = min(D, 12) 
    idx = rng.choice(D, size=k, replace=False)
    def freq_warp(seg):
        tt = np.arange(seg.shape[0])
        seg[:, idx] = 0.5*seg[:, idx] + 1.0*np.sin(2*np.pi*0.45*tt)[:, None]
        return seg
    inject_segment(s, s+width, freq_warp)

print(f"[DATA] Train={X_train.shape}  Test={X_test.shape}  Anom%={labels.mean():.2%}")

# Standardlization
scaler = StandardScaler().fit(X_train)
X_tr_std = scaler.transform(X_train).astype(np.float32)
X_te_std = scaler.transform(X_test).astype(np.float32)

# Sliding window（train stride=10, test stride=1)
SEQ_LEN = 100
STRIDE_TRAIN = 10

def get_subseqs(arr, seq_len, stride):
    T, d = arr.shape
    idx = np.arange(0, T-seq_len+1, stride)
    out = np.empty((len(idx), seq_len, d), dtype=np.float32)
    for k, s in enumerate(idx):
        out[k] = arr[s:s+seq_len]
    return out, idx

Xtr_win, _       = get_subseqs(X_tr_std, SEQ_LEN, STRIDE_TRAIN)
Xte_win, starts  = get_subseqs(X_te_std, SEQ_LEN, 1)   # stride=1


# IF uses a flat vector
Xtr_flat = Xtr_win.reshape(len(Xtr_win), -1)
Xte_flat = Xte_win.reshape(len(Xte_win), -1)

# Window label (marking 1 if any anomaly is covered)
win_labels = np.zeros(len(starts), dtype=np.int32)
for i, s in enumerate(starts):
    if labels[s:s+SEQ_LEN].max() == 1:
        win_labels[i] = 1



# Point-level alignment（Restore the window score to the timestamp and create point-level indicators)
# point_scores = np.zeros(T_test, dtype=np.float32)
# counts = np.zeros(T_test, dtype=np.int32)
# for s, v in zip(starts, score_dif):
#     point_scores[s:s+SEQ_LEN] = np.maximum(point_scores[s:s+SEQ_LEN], v)
#     counts[s:s+SEQ_LEN] += 1
# mask = counts > 0
# auc_point = roc_auc_score(labels[mask], point_scores[mask])
# pr_point  = average_precision_score(labels[mask], point_scores[mask])
# print(f"[DIF|point] AUC-ROC={auc_point:.4f}  AUC-PR={pr_point:.4f}  (covered_points={mask.sum()})")


[DATA] Train=(20000, 3)  Test=(40000, 3)  Anom%=1.00%


In [ ]:
# DIF（ts：dilated_conv)
# The parameters are based on the paper
dif = DIF(
    data_type='ts',
    network_name='dilated_conv',
    new_ensemble_method=0,
    batch_size=64,
    rep_dim=20,
    layers=2,
    hidden_dim=32,
    n_ensemble=50,        
    n_estimators=6,
    random_state=42
)

dif.fit(Xtr_win)                               
score_dif_raw = dif.decision_function(Xte_win) 

# Direction
auc_raw = roc_auc_score(win_labels, score_dif_raw)
auc_neg = roc_auc_score(win_labels, -score_dif_raw)
score_dif = score_dif_raw if auc_raw >= auc_neg else -score_dif_raw

auc_dif = roc_auc_score(win_labels, score_dif)
pr_dif  = average_precision_score(win_labels, score_dif)
print(f"[DIF]   AUC-ROC={auc_dif:.4f}  AUC-PR={pr_dif:.4f}  (wins={len(score_dif)})")


network additional parameters: {'layers': 2, 'hidden_dim': 32, 'n_emb': 20}


testing ensemble process: 100%|█████████████████████████████████████| 50/50 [00:23<00:00,  2.15it/s]


[DIF]   AUC-ROC=0.9856  AUC-PR=0.9576  (wins=39901)


In [3]:
# IF
ifor = IsolationForest(
    n_estimators=300,  
    max_samples=256,
    contamination='auto',
    random_state=42,
    n_jobs=-1
)

ifor.fit(Xtr_flat)
score_if = -ifor.score_samples(Xte_flat)
auc_if = roc_auc_score(win_labels, score_if)
pr_if  = average_precision_score(win_labels, score_if)
print(f"[IForest]  AUC-ROC={auc_if:.4f}  AUC-PR={pr_if:.4f}  (wins={len(score_if)})")

[IForest]  AUC-ROC=0.9610  AUC-PR=0.8666  (wins=39901)
